# Teks — Topic Modeling lengkap (pilih metode + jumlah topik, lalu dokumentasi untuk slide)
**Kapan dipakai:** soal meminta topik-topik utama dari kumpulan teks, dengan atau tanpa label.
Semua metode di sini dilatih sendiri di data lomba (tidak ada zero-shot / few-shot).

**Langkah:**
1. Bersihkan teks khusus topik (stopword, angka, tautan).
2. Tiga metode di kosakata yang sama: NMF, LDA, dan embedding + UMAP + KMeans + c-TF-IDF
   (cara kerja BERTopic; versi pustaka dengan HDBSCAN ada di `topik_bertopic`).
3. Tiap metode x jumlah topik dinilai TANPA label: koherensi NPMI, keragaman kata, kestabilan
   antar seed. Yang terbaik dipakai (atau paksa lewat `FINAL_METHOD` / `FINAL_TOPICS`).
4. Dokumentasi: kata teratas, ukuran topik, awan kata, topik x kelas, tren waktu, peta 2D.

**Keluaran** di `OUTPUT_FOLDER`: PNG + tabel ringkas + `fakta_topik.md` (boleh dibawa untuk
slide), dan `topik_dokumen.csv` (topik per baris: TETAP di komputer lomba).

**Data demo:** SmSA (ulasan Indonesia, 3 kelas). Label hanya dipakai di bagian "topik x kelas".

In [ ]:
import os
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import normalized_mutual_info_score
from sklearn.model_selection import cross_val_score
from umap import UMAP
from wordcloud import WordCloud

## Konfigurasi

In [ ]:
# ============ NILAI SOAL: isi SAMA di semua notebook 1_..7_ (tidak ada konfig.py) ============
# Sesudah mengubah: Restart Kernel + Run All. Nilai awal = data demo SmSA (sentimen, 3 kelas).
# ---- File soal (garis miring /, bukan \) ----
# ---- Kolom ----
ID_COL = "id"                                 # GANTI: kolom id
TEXT_COLS = ["text"]                          # GANTI: kolom teks. 4_finetune hanya memakai yang pertama
TARGET = "label"                              # GANTI: kolom kelas
SUBMIT_COL = TARGET                           # GANTI bila kolom prediksi di sample bernama lain
LABEL_COLS = None                             # MULTILABEL: daftar kolom label 0/1, mis. ["a", "b"].
#                                               None = single label. Bila diisi: pakai 3m, 4m, 5m
DATE_COL = None                               # kolom tanggal (tren di EDA dan topik), None = tidak
LANGUAGE = "id"                               # "id" = stopword Indonesia, "en" = Inggris
# ---- Hasil ----
HASIL = "../../outputs/final"                 # GANTI NAMA bila ganti dataset (mis. "../../outputs/lomba"),
#                                               supaya OOF demo tidak tercampur dengan OOF soal
TOPIK_BERAT = True                            # True = kata topik 6_topik dari kolom <teks>_tfidf (slang -> baku,
#                                               stopword, stemming). Embedding tetap teks ringan. False = teks ringan

# Coba jalur multilabel di data demo CASA (ulasan mobil, 6 label): buka # di empat baris ini
# TRAIN_MENTAH = "demo/casa_train.csv"
# TEST_MENTAH = "demo/casa_test.csv"
# LABEL_COLS = ["fuel", "machine", "others", "part", "price", "service"]
# HASIL = "../../outputs/final_multilabel"
# Multilabel: EDA, praproses, dan "topik x kelas" butuh SATU kolom kelas. Bila TARGET bukan salah satu
# LABEL_COLS, dipakai label pertama (ganti TARGET ke label yang paling penting).
if LABEL_COLS is not None and TARGET not in LABEL_COLS:
    TARGET = LABEL_COLS[0]
    SUBMIT_COL = TARGET
# ====================================================================================================

EMBEDDING_MODEL = "../../pretrained/minilm"                 # multibahasa, cepat
# EMBEDDING_MODEL = "../../pretrained/e5_base"              # lebih kuat, lebih lambat

EXTRA_STOPWORDS = []             # GANTI: kata umum di data lomba yang mengotori topik

METHODS = ["nmf", "lda", "embedding"]     # hapus yang tidak mau dicoba (lda paling lambat)
TOPIC_COUNTS = [4, 6, 8, 10, 12, 15]      # jumlah topik yang dibandingkan
SEEDS = [42, 7, 2026]                     # kestabilan = kemiripan topik antar seed
FINAL_METHOD = None              # None = pilih otomatis, atau "nmf" / "lda" / "embedding"
FINAL_TOPICS = None              # None = pilih otomatis, atau angka, mis. 8
MIN_DIVERSITY = 0.7              # calon dengan keragaman kata di bawah ini tidak dipilih

NGRAM = (1, 2)                   # (1, 1) = kata tunggal saja
MIN_DF = 5                       # kata harus muncul di >= MIN_DF teks
MAX_DF = 0.4                     # kata di > 40% teks dianggap terlalu umum
N_TOP_WORDS = 10
AMAN_NDA = True                  # True = tidak mencetak teks asli, kata minimal di 5 teks
N_EXAMPLES = 3                   # contoh teks per topik (hanya bila AMAN_NDA = False)
MAP_SAMPLE = 5000                # titik di peta 2D
SEED = 42

# ---- jalur final-pipeline: dihitung dari HASIL di atas, tidak perlu diubah ----
TRAIN_CSV = f"{HASIL}/praproses/train_bersih.csv"
TEST_CSV = f"{HASIL}/praproses/test_bersih.csv"
OUTPUT_FOLDER = f"{HASIL}/topik"
TOPIC_TEXT_COLS = TEXT_COLS
if TOPIK_BERAT:
    TOPIC_TEXT_COLS = []
    for column in TEXT_COLS:
        TOPIC_TEXT_COLS.append(column + "_tfidf")
EXTRA_STOPWORDS += ["urltoken", "mentiontoken", "angkatoken", "emoji", "banget", "enggak"]   # token buatan + KEEP_WORDS 2_praproses

## Alat simpan
Tiap grafik disimpan sebagai PNG + tabel ringkasnya; tiap temuan dicatat untuk `fakta_topik.md`.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False})
facts = {}
saved = []
if AMAN_NDA:
    MIN_DF = max(MIN_DF, 5)


def add_fact(section, text):
    facts.setdefault(section, []).append(text)
    print("-", text)


def save_figure(figure, name, table):
    figure.savefig(f"{OUTPUT_FOLDER}/{name}.png")
    table.to_csv(f"{OUTPUT_FOLDER}/{name}.csv", index=False)
    saved.append((name, table))
    plt.show()


def pct(value):
    return f"{100 * value:.1f}%".replace(".", ",")


print("alat simpan siap:", OUTPUT_FOLDER, "| AMAN_NDA:", AMAN_NDA, "| MIN_DF:", MIN_DF)

## Baca data
Topik dicari di train + test sekaligus (tanpa label, jadi tidak bocor): topik yang hanya ada
di test ikut terlihat.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
train["asal"] = "train"
data = train
if TEST_CSV is not None:
    test = pd.read_csv(TEST_CSV)
    test["asal"] = "test"
    data = pd.concat([train, test], ignore_index=True)
if ID_COL is None:
    ID_COL = "baris"
    data[ID_COL] = np.arange(len(data))
raw_text = data[TEXT_COLS[0]].fillna("").astype(str)
for column in TEXT_COLS[1:]:
    raw_text = raw_text + " " + data[column].fillna("").astype(str)
add_fact("Data", f"{len(data)} teks dipakai untuk mencari topik "
         f"({(data['asal'] == 'train').sum()} train, {(data['asal'] == 'test').sum()} test).")

## Bersihkan teks khusus topik
Topik dibaca dari kata isi. Tautan, sebutan akun, angka, dan stopword dibuang; teks asli tetap
dipakai untuk embedding. `TOPIK_BERAT = True` (sel Konfigurasi): kata diambil dari kolom `<teks>_tfidf`
2_praproses (slang -> baku, stemming), jadi "gak"/"tidak" dan "makanannya"/"makan" menyatu.

In [ ]:
stop_words = set()
if LANGUAGE == "id":
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    stop_words = set(StopWordRemoverFactory().get_stop_words())
if LANGUAGE == "en":
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    stop_words = set(ENGLISH_STOP_WORDS)
stop_words = sorted(stop_words | set(EXTRA_STOPWORDS))


def clean_for_topics(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+|@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


topic_text = data[TOPIC_TEXT_COLS[0]].fillna("").astype(str)   # TOPIK_BERAT di sel Konfigurasi
for column in TOPIC_TEXT_COLS[1:]:
    topic_text = topic_text + " " + data[column].fillna("").astype(str)
clean_text = topic_text.map(clean_for_topics)
print("kolom kata topik:", TOPIC_TEXT_COLS)
print("stopword:", len(stop_words), "| teks kosong sesudah dibersihkan:", (clean_text == "").sum())

## Kosakata
Satu kosakata untuk semua metode supaya skornya bisa dibandingkan. Daftar kata paling sering
di bawah = bahan `EXTRA_STOPWORDS` (kata umum yang tidak membedakan topik).

In [ ]:
word_counter = CountVectorizer(stop_words=stop_words, ngram_range=NGRAM, min_df=MIN_DF,
                               max_df=MAX_DF, token_pattern=r"(?u)\b[a-z]{3,}\b")
counts = word_counter.fit_transform(clean_text)
vocabulary = word_counter.get_feature_names_out()
tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(counts)
presence = (counts > 0).astype(np.float32).tocsc()
doc_frequency = np.asarray(presence.sum(axis=0)).ravel()
n_docs = counts.shape[0]
empty_docs = int((counts.sum(axis=1) == 0).sum())
add_fact("Data", f"Kosakata {len(vocabulary)} kata/frasa (muncul di >= {MIN_DF} teks dan "
         f"<= {pct(MAX_DF)} teks), {empty_docs} teks tidak punya kata di kosakata.")
print("paling sering:", ", ".join(vocabulary[np.argsort(-doc_frequency)[:30]]))

## Ukuran mutu tanpa label
- **Koherensi NPMI** (-1 .. 1): kata-kata teratas satu topik sering muncul di teks yang sama?
  Makin tinggi makin mudah diberi nama.
- **Keragaman** (0 .. 1): bagian kata teratas yang unik. Rendah = topik saling mengulang.
- **Kestabilan** (0 .. 1): topik yang sama muncul lagi bila seed diganti (Jaccard kata teratas).

In [ ]:
def top_words(topic_word):
    return np.argsort(-topic_word, axis=1)[:, :N_TOP_WORDS]


def topic_npmi(word_index):
    together = (presence[:, word_index].T @ presence[:, word_index]).toarray() / n_docs
    alone = doc_frequency[word_index] / n_docs
    values = []
    for first in range(len(word_index)):
        for second in range(first + 1, len(word_index)):
            joint = together[first, second]
            if joint == 0:
                values.append(-1.0)
                continue
            pmi = np.log(joint / (alone[first] * alone[second]))
            values.append(pmi / max(-np.log(joint), 1e-9))
    return float(np.mean(values))


def diversity(tops):
    return len(np.unique(tops)) / tops.size


demo_tops = np.argsort(-doc_frequency)[:N_TOP_WORDS]
print("NPMI 10 kata tersering (pembanding, bukan topik):", round(topic_npmi(demo_tops), 3))

In [ ]:
def stability(tops_a, tops_b):
    best = []
    for row_a in tops_a:
        scores = []
        for row_b in tops_b:
            shared = len(set(row_a) & set(row_b))
            scores.append(shared / (2 * N_TOP_WORDS - shared))
        best.append(max(scores))
    return float(np.mean(best))


print("uji: topik identik =", stability(demo_tops[None, :], demo_tops[None, :]))

## Embedding kalimat (hanya untuk metode "embedding")
Dihitung sekali dari teks asli, lalu diperkecil UMAP per seed. Model dimuat dari folder lokal.

In [ ]:
embeddings = None
reduced_cache = {}
if "embedding" in METHODS:
    from sentence_transformers import SentenceTransformer
    start = time.time()
    encoder = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = encoder.encode(list(raw_text), batch_size=64, show_progress_bar=False,
                                normalize_embeddings=True)
    print("embedding:", embeddings.shape, f"| {time.time() - start:.0f} detik")


def reduce_embeddings(seed):
    if seed not in reduced_cache:
        reducer = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                       random_state=seed)
        reduced_cache[seed] = reducer.fit_transform(embeddings)
    return reduced_cache[seed]


print("metode:", METHODS)

## Tiga metode, satu bentuk keluaran
Tiap fungsi mengembalikan `doc_topic` (baris = teks, kolom = topik) dan `topic_word`
(baris = topik, kolom = kosakata), jadi bagian sesudahnya sama untuk semua metode.

In [ ]:
def fit_nmf(n_topics, seed):
    model = NMF(n_components=n_topics, init="nndsvdar", random_state=seed, max_iter=400)
    doc_topic = model.fit_transform(tfidf)
    return doc_topic, model.components_


def fit_lda(n_topics, seed):
    model = LatentDirichletAllocation(n_components=n_topics, learning_method="online",
                                      max_iter=15, random_state=seed, n_jobs=1)
    doc_topic = model.fit_transform(counts)
    return doc_topic, model.components_


def class_tfidf(groups, n_topics):
    # c-TF-IDF BERTopic: kata khas tiap kelompok dibanding kelompok lain
    indicator = sparse.csr_matrix((np.ones(n_docs), (groups, np.arange(n_docs))),
                                  shape=(n_topics, n_docs))
    group_counts = np.asarray((indicator @ counts).todense())
    frequency = group_counts / np.maximum(group_counts.sum(axis=1, keepdims=True), 1)
    average = group_counts.sum() / n_topics
    return frequency * np.log(1 + average / np.maximum(group_counts.sum(axis=0), 1))


print("fit_nmf, fit_lda, class_tfidf siap")

In [ ]:
def fit_embedding(n_topics, seed):
    clusterer = KMeans(n_clusters=n_topics, n_init=10, random_state=seed)
    groups = clusterer.fit_predict(reduce_embeddings(seed))
    doc_topic = np.zeros((n_docs, n_topics))
    doc_topic[np.arange(n_docs), groups] = 1.0
    return doc_topic, class_tfidf(groups, n_topics)


FITTERS = {"nmf": fit_nmf, "lda": fit_lda, "embedding": fit_embedding}


def score_run(method, n_topics):
    npmi_values, diversity_values, tops_per_seed = [], [], []
    for seed in SEEDS:
        tops = top_words(FITTERS[method](n_topics, seed)[1])
        coherence = 0.0
        for row in tops:
            coherence += topic_npmi(row) / n_topics
        npmi_values.append(coherence)
        diversity_values.append(diversity(tops))
        tops_per_seed.append(tops)
    stable = [np.nan] * (len(SEEDS) == 1)
    for position in range(1, len(tops_per_seed)):
        stable.append(stability(tops_per_seed[0], tops_per_seed[position]))
    return np.mean(npmi_values), np.mean(diversity_values), np.mean(stable)


print("score_run siap")

## Bandingkan metode x jumlah topik
Bagian terlama di notebook ini. Waktu mepet: kurangi `METHODS`, `TOPIC_COUNTS`, atau `SEEDS`.

In [ ]:
score_rows = []
for method in METHODS:
    for n_topics in TOPIC_COUNTS:
        start = time.time()
        npmi, diverse, stable = score_run(method, n_topics)
        score_rows.append({"metode": method, "jumlah_topik": n_topics, "npmi": round(npmi, 4),
                           "keragaman": round(diverse, 3), "kestabilan": round(stable, 3),
                           "detik": round(time.time() - start, 1)})
        print(score_rows[-1])
scores = pd.DataFrame(score_rows)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = {"npmi": "Koherensi NPMI (tinggi = topik mudah dinamai)",
          "keragaman": "Keragaman kata (tinggi = topik tidak mengulang)",
          "kestabilan": "Kestabilan antar seed (tinggi = topik konsisten)"}
for axis, column in zip(axes, titles):
    for method in METHODS:
        part = scores[scores["metode"] == method]
        axis.plot(part["jumlah_topik"], part[column], marker="o", label=method)
    axis.set_title(titles[column], fontsize=10)
    axis.set_xlabel("jumlah topik")
    axis.set_xticks(TOPIC_COUNTS)
axes[0].legend(frameon=False)
save_figure(figure, "01_pilih_metode_dan_jumlah_topik", scores)

## Pilih metode dan jumlah topik
Aturan: NPMI tertinggi di antara calon yang keragamannya >= `MIN_DIVERSITY`. Bila pilihan
otomatis terasa kurang pas saat membaca kata teratas, paksa lewat `FINAL_METHOD` /
`FINAL_TOPICS` di Konfigurasi dan tulis alasannya di slide.

In [ ]:
candidates = scores[scores["keragaman"] >= MIN_DIVERSITY]
if len(candidates) == 0:
    candidates = scores
if FINAL_METHOD is not None:
    candidates = scores[scores["metode"] == FINAL_METHOD]
if FINAL_TOPICS is not None:
    candidates = candidates[candidates["jumlah_topik"] == FINAL_TOPICS]
if len(candidates) == 0:
    candidates = pd.DataFrame([{"metode": FINAL_METHOD, "jumlah_topik": FINAL_TOPICS,
                                "npmi": np.nan, "keragaman": np.nan, "kestabilan": np.nan}])
chosen = candidates.sort_values("npmi", ascending=False).iloc[0]
chosen_method = chosen["metode"]
chosen_topics = int(chosen["jumlah_topik"])
for method in METHODS:
    best = scores[scores["metode"] == method].sort_values("npmi", ascending=False).iloc[0]
    add_fact("Pemilihan model", f"{method}: NPMI terbaik {best['npmi']:.3f} di "
             f"{int(best['jumlah_topik'])} topik (keragaman {best['keragaman']:.2f}, "
             f"kestabilan {best['kestabilan']:.2f}).")
add_fact("Pemilihan model", f"Dipakai: {chosen_method} dengan {chosen_topics} topik "
         f"(NPMI {chosen['npmi']:.3f}, keragaman {chosen['keragaman']:.2f}, "
         f"kestabilan {chosen['kestabilan']:.2f}).")

## Model akhir

In [ ]:
doc_topic, topic_word = FITTERS[chosen_method](chosen_topics, SEED)
tops = top_words(topic_word)
row_total = np.maximum(doc_topic.sum(axis=1, keepdims=True), 1e-12)
doc_share = doc_topic / row_total
data["topik"] = doc_share.argmax(axis=1)
data["kuat_topik"] = doc_share.max(axis=1).round(3)
data.loc[doc_topic.sum(axis=1) == 0, "topik"] = -1      # teks tanpa kata di kosakata
topic_rows = []
for topic in range(chosen_topics):
    words = ", ".join(vocabulary[tops[topic]])
    size = int((data["topik"] == topic).sum())
    topic_rows.append({"topik": topic, "jumlah_teks": size, "porsi": round(size / n_docs, 4),
                       "npmi": round(topic_npmi(tops[topic]), 3), "kata_teratas": words})
topic_table = pd.DataFrame(topic_rows)
pd.set_option("display.max_colwidth", 200)
topic_table

## Beri nama topik
Baca `kata_teratas` di tabel atas, lalu isi nama pendek per nomor topik. Yang tidak diisi
memakai tiga kata teratasnya. Nama ini dipakai di semua grafik dan di `fakta_topik.md`.

In [ ]:
TOPIC_NAMES = {}                 # GANTI: {0: "harga dan promo", 1: "pelayanan", ...}

names = []
for topic in range(chosen_topics):
    fallback = " / ".join(vocabulary[tops[topic][:3]])
    names.append(f"T{topic} " + TOPIC_NAMES.get(topic, fallback))
topic_table["nama"] = names
untopiced = int((data["topik"] == -1).sum())
biggest = topic_table.sort_values("jumlah_teks", ascending=False).iloc[0]
weakest = topic_table.sort_values("npmi").iloc[0]
add_fact("Topik", f"Topik terbesar: {biggest['nama']} ({pct(biggest['porsi'])} teks). "
         f"{untopiced} teks tidak mendapat topik.")
add_fact("Topik", f"Topik paling kabur (NPMI terendah {weakest['npmi']:.3f}): {weakest['nama']}, "
         "calon digabung atau dicatat sebagai topik campuran.")
for row in topic_rows:
    add_fact("Topik", f"{names[row['topik']]}: {pct(row['porsi'])} teks, NPMI {row['npmi']:.3f}, "
             f"kata: {row['kata_teratas']}.")

## Ukuran tiap topik

In [ ]:
ordered = topic_table.sort_values("jumlah_teks")
figure, axis = plt.subplots(figsize=(9, 0.45 * chosen_topics + 1.5))
bars = axis.barh(ordered["nama"], ordered["jumlah_teks"], color="#3d6fa8")
axis.bar_label(bars, labels=list(ordered["porsi"].map(pct)), padding=3, fontsize=9)
axis.set_xlabel("jumlah teks")
axis.set_title(f"Ukuran {chosen_topics} topik ({chosen_method}), n = {n_docs}")
save_figure(figure, "02_ukuran_topik", topic_table)

## Kata teratas tiap topik
Batang = bobot kata di topik itu (skala tiap panel sendiri).

In [ ]:
columns = min(4, chosen_topics)
rows = int(np.ceil(chosen_topics / columns))
figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 2.8 * rows), squeeze=False)
word_rows = []
for topic in range(chosen_topics):
    axis = axes[topic // columns][topic % columns]
    weights = topic_word[topic, tops[topic]]
    axis.barh(vocabulary[tops[topic]][::-1], weights[::-1], color="#3d6fa8")
    axis.set_title(names[topic], fontsize=9)
    axis.set_xticks([])
    for position in range(N_TOP_WORDS):
        word_rows.append({"topik": names[topic], "urutan": position + 1,
                          "kata": vocabulary[tops[topic][position]],
                          "bobot": round(float(weights[position]), 4)})
for spare in range(chosen_topics, rows * columns):
    axes[spare // columns][spare % columns].axis("off")
figure.tight_layout()
save_figure(figure, "03_kata_teratas", pd.DataFrame(word_rows))

## Awan kata tiap topik

In [ ]:
figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 2.6 * rows), squeeze=False)
for topic in range(chosen_topics):
    axis = axes[topic // columns][topic % columns]
    weights = {}
    for index in np.argsort(-topic_word[topic])[:40]:
        weights[vocabulary[index]] = float(topic_word[topic, index]) + 1e-9
    cloud = WordCloud(width=600, height=380, background_color="white", colormap="viridis",
                      random_state=SEED).generate_from_frequencies(weights)
    axis.imshow(cloud)
    axis.set_title(names[topic], fontsize=9)
for spare in range(rows * columns):
    axes[spare // columns][spare % columns].axis("off")
figure.tight_layout()
save_figure(figure, "04_awan_kata", topic_table[["nama", "kata_teratas"]])

## Topik yang mirip
Kemiripan cosine antar topik di ruang kata. Pasangan di atas 0,5 = calon digabung (turunkan
jumlah topik) atau perlu dijelaskan bedanya di slide.

In [ ]:
unit = topic_word / np.maximum(np.linalg.norm(topic_word, axis=1, keepdims=True), 1e-12)
similarity = unit @ unit.T
figure, axis = plt.subplots(figsize=(0.55 * chosen_topics + 4, 0.5 * chosen_topics + 2.5))
image = axis.imshow(similarity, cmap="Blues", vmin=0, vmax=1)
axis.set_xticks(range(chosen_topics), range(chosen_topics))
axis.set_yticks(range(chosen_topics), names, fontsize=8)
figure.colorbar(image, ax=axis, shrink=0.8, label="kemiripan")
pair_rows = []
for first in range(chosen_topics):
    for second in range(first + 1, chosen_topics):
        pair_rows.append({"topik_a": names[first], "topik_b": names[second],
                          "kemiripan": round(float(similarity[first, second]), 3)})
pairs = pd.DataFrame(pair_rows).sort_values("kemiripan", ascending=False)
closest = pairs.iloc[0]
add_fact("Topik", f"Pasangan topik paling mirip: {closest['topik_a']} dan {closest['topik_b']} "
         f"(cosine {closest['kemiripan']:.2f}), {(pairs['kemiripan'] > 0.5).sum()} pasangan > 0,5.")
save_figure(figure, "05_kemiripan_topik", pairs)

## Topik x kelas (hanya bila `TARGET` ada)
Tiap sel = porsi kelas di dalam satu topik (baris berjumlah 100%). **Lift** = porsi itu dibagi
porsi kelas di seluruh train: > 1 berarti topik itu condong ke kelas tersebut.

In [ ]:
labeled = data[data["asal"] == "train"]
has_target = TARGET is not None and TARGET in data.columns
if has_target:
    labeled = labeled[labeled[TARGET].notna() & (labeled["topik"] >= 0)]
    share = pd.crosstab(labeled["topik"], labeled[TARGET], normalize="index")
    base = labeled[TARGET].value_counts(normalize=True)
    lift = share / base[share.columns]
    figure, axis = plt.subplots(figsize=(1.3 * share.shape[1] + 4, 0.5 * len(share) + 2))
    image = axis.imshow(share.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
    axis.set_xticks(range(share.shape[1]), share.columns)
    axis.set_yticks(range(len(share)), np.array(names)[share.index], fontsize=8)
    for row in range(len(share)):
        for column in range(share.shape[1]):
            value = share.iloc[row, column]
            axis.text(column, row, f"{100 * value:.0f}%\nx{lift.iloc[row, column]:.1f}",
                      ha="center", va="center", fontsize=8,
                      color=["black", "white"][int(value > 0.6)])
    axis.set_title("Porsi kelas tiap topik (angka kedua = lift)")
    table = share.round(3).reset_index()
    table["topik"] = np.array(names)[share.index]
    save_figure(figure, "06_topik_x_kelas", table)
print("TARGET dipakai:", has_target)

In [ ]:
if has_target:
    for name in share.columns:
        top_topic = lift[name].idxmax()
        add_fact("Topik x kelas", f"Kelas {name} (porsi umum {pct(base[name])}) paling kuat di "
                 f"{names[top_topic]}: {pct(share.loc[top_topic, name])} dari topik itu "
                 f"(lift {lift.loc[top_topic, name]:.1f}).")
    nmi = normalized_mutual_info_score(labeled[TARGET], labeled["topik"])
    add_fact("Topik x kelas", f"Keterkaitan topik dengan kelas: NMI {nmi:.3f} "
             "(0 = tidak terkait, 1 = topik sama persis dengan kelas).")
    features = doc_share[labeled.index.to_numpy()]
    topic_only = cross_val_score(LogisticRegression(max_iter=2000, class_weight="balanced"),
                                 features, labeled[TARGET], cv=5, scoring="f1_macro").mean()
    add_fact("Topik x kelas", f"Menebak kelas hanya dari proporsi topik: F1 makro {topic_only:.3f} "
             f"(5 fold, {len(base)} kelas). Rendah = topik dan kelas memberi informasi berbeda.")
    # fitur tambahan untuk baseline_oof: np.save(f"{OUTPUT_FOLDER}/fitur_topik.npy", doc_share)

## Topik x label (hanya soal MULTILABEL, `LABEL_COLS` di sel Konfigurasi)
Tiap sel = porsi teks di satu topik yang punya label itu (satu teks boleh punya beberapa label,
jadi baris TIDAK berjumlah 100%). **Lift** = porsi itu dibagi porsi label di seluruh train.

In [ ]:
if LABEL_COLS is not None:
    multi = data[(data["asal"] == "train") & (data["topik"] >= 0)]
    label_share = multi.groupby("topik")[LABEL_COLS].mean()
    label_base = multi[LABEL_COLS].mean()
    label_lift = label_share / label_base.clip(lower=1e-9)
    figure, axis = plt.subplots(figsize=(1.1 * len(LABEL_COLS) + 4, 0.5 * len(label_share) + 2))
    axis.imshow(label_share.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
    axis.set_xticks(range(len(LABEL_COLS)), LABEL_COLS, rotation=30, ha="right")
    axis.set_yticks(range(len(label_share)), np.array(names)[label_share.index], fontsize=8)
    for row in range(len(label_share)):
        for column in range(len(LABEL_COLS)):
            value = label_share.iloc[row, column]
            axis.text(column, row, f"{100 * value:.0f}%\nx{label_lift.iloc[row, column]:.1f}",
                      ha="center", va="center", fontsize=8, color=["black", "white"][int(value > 0.6)])
    axis.set_title("Porsi teks berlabel tiap topik (angka kedua = lift)")
    table = label_share.round(3).reset_index()
    table["topik"] = np.array(names)[label_share.index]
    save_figure(figure, "06b_topik_x_label", table)
    for name in LABEL_COLS:
        top_topic = label_lift[name].idxmax()
        add_fact("Topik x label", f"Label {name} (porsi umum {pct(label_base[name])}) paling kuat di "
                 f"{names[top_topic]}: {pct(label_share.loc[top_topic, name])} dari topik itu "
                 f"(lift {label_lift.loc[top_topic, name]:.1f}).")
print("LABEL_COLS:", LABEL_COLS)

## Topik train vs test
Topik yang porsinya jauh lebih besar di test = isi test bergeser dari train.

In [ ]:
origin_share = pd.crosstab(data["topik"], data["asal"], normalize="columns")
origin_share = origin_share[origin_share.index >= 0]
if "test" in origin_share.columns:
    figure, axis = plt.subplots(figsize=(9, 0.5 * chosen_topics + 1.5))
    positions = np.arange(len(origin_share))
    axis.barh(positions + 0.2, origin_share["train"], height=0.4, label="train", color="#3d6fa8")
    axis.barh(positions - 0.2, origin_share["test"], height=0.4, label="test", color="#e08a3c")
    axis.set_yticks(positions, np.array(names)[origin_share.index], fontsize=8)
    axis.set_xlabel("porsi teks")
    axis.legend(frameon=False)
    gap = (origin_share["test"] - origin_share["train"]).abs()
    add_fact("Train vs test", f"Beda porsi topik terbesar antara train dan test: "
             f"{names[gap.idxmax()]} ({pct(origin_share.loc[gap.idxmax(), 'train'])} vs "
             f"{pct(origin_share.loc[gap.idxmax(), 'test'])}).")
    table = origin_share.round(4).reset_index()
    table["topik"] = np.array(names)[origin_share.index]
    save_figure(figure, "07_topik_train_vs_test", table)

## Tren topik per bulan (hanya bila `DATE_COL` ada)

In [ ]:
if DATE_COL is not None:
    dated = data[data["topik"] >= 0].copy()
    dated["bulan"] = pd.to_datetime(dated[DATE_COL], errors="coerce").dt.to_period("M")
    dated = dated[dated["bulan"].notna()]
    trend = pd.crosstab(dated["bulan"], dated["topik"], normalize="index")
    figure, axis = plt.subplots(figsize=(11, 4.5))
    for topic in trend.columns:
        axis.plot(trend.index.astype(str), trend[topic], marker="o", markersize=3,
                  label=names[topic])
    axis.set_ylabel("porsi teks bulan itu")
    axis.tick_params(axis="x", rotation=60, labelsize=8)
    axis.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
    change = trend.iloc[-1] - trend.iloc[0]
    add_fact("Tren waktu", f"{trend.index[0]} sampai {trend.index[-1]}: paling naik "
             f"{names[change.idxmax()]} ({change.max():+.1%}), paling turun "
             f"{names[change.idxmin()]} ({change.min():+.1%}).")
    table = trend.round(4).reset_index()
    table["bulan"] = table["bulan"].astype(str)
    save_figure(figure, "08_tren_topik", table)
print("DATE_COL:", DATE_COL)

## Peta 2D dokumen
Tiap titik = satu teks, warna = topik. Kelompok yang terpisah jelas = topik tegas; yang
bertumpuk = topik berbagi kosakata. Peta memakai TF-IDF (atau embedding bila ada).

In [ ]:
rng = np.random.default_rng(SEED)
picked = np.sort(rng.choice(n_docs, size=min(MAP_SAMPLE, n_docs), replace=False))
if embeddings is not None:
    map_source = embeddings[picked]
if embeddings is None:
    from sklearn.decomposition import TruncatedSVD
    map_source = TruncatedSVD(n_components=50, random_state=SEED).fit_transform(tfidf[picked])
points = UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED)
points = points.fit_transform(map_source)
picked_topic = data["topik"].to_numpy()[picked]
figure, axis = plt.subplots(figsize=(9, 7))
palette = plt.get_cmap("tab20")(np.r_[0:20:2, 1:20:2])   # warna tua dulu
center_rows = []
for topic in range(chosen_topics):
    mask = picked_topic == topic
    axis.scatter(points[mask, 0], points[mask, 1], s=5, alpha=0.5, color=palette[topic % 20],
                 label=names[topic])
    if mask.sum() > 0:
        center_rows.append({"topik": names[topic], "x": round(float(points[mask, 0].mean()), 2),
                            "y": round(float(points[mask, 1].mean()), 2), "titik": int(mask.sum())})
axis.set_xticks([])
axis.set_yticks([])
axis.legend(frameon=False, fontsize=8, markerscale=3, bbox_to_anchor=(1.01, 1), loc="upper left")
axis.set_title(f"Peta {len(picked)} teks, warna = topik")
save_figure(figure, "09_peta_dokumen", pd.DataFrame(center_rows))

## Contoh teks per topik
Mati selama `AMAN_NDA = True` (teks asli tidak boleh keluar dari komputer lomba). Nyalakan
hanya untuk membaca di layar saat memberi nama topik, lalu kosongkan output sel ini.

In [ ]:
if not AMAN_NDA:
    for topic in range(chosen_topics):
        part = data[data["topik"] == topic].sort_values("kuat_topik", ascending=False)
        print("\n==", names[topic])
        for text in raw_text[part.index[:N_EXAMPLES]]:
            print("   -", text[:200])
print("contoh teks dicetak:", not AMAN_NDA)

## Simpan topik per baris (tetap di komputer lomba)
`topik_dokumen.csv` berisi satu baris per teks, jadi TIDAK ikut dibawa keluar. Kolom `topik`
dan `nama_topik` bisa disalin ke file kumpul bila panitia meminta topik per teks.

In [ ]:
result = pd.DataFrame()
result[ID_COL] = data[ID_COL]
result["asal"] = data["asal"]
result["topik"] = data["topik"]
name_of = dict(zip(range(chosen_topics), names))
name_of[-1] = "tanpa topik"
result["nama_topik"] = result["topik"].map(name_of)
result["kuat_topik"] = data["kuat_topik"]
result.to_csv(f"{OUTPUT_FOLDER}/topik_dokumen.csv", index=False)
topic_table.to_csv(f"{OUTPUT_FOLDER}/info_topik.csv", index=False)
print(result["nama_topik"].value_counts())

## Fakta untuk slide
`fakta_topik.md` = semua temuan + tabel di balik tiap grafik, tanpa teks asli. Isinya juga
dicetak di bawah supaya ikut tersimpan di output notebook.

In [ ]:
lines = ["# Fakta topic modeling (otomatis dari topik_lengkap)", "",
         f"Dibuat {time.strftime('%Y-%m-%d %H:%M')}. Metode dilatih sendiri di data lomba.",
         "Untuk AI pembuat slide: semua angka di bawah dihitung dari data.",
         "Jangan menambah angka, contoh teks, atau nama topik yang tidak tertulis di sini.", ""]
for section, items in facts.items():
    lines.append(f"## {section}")
    for item in items:
        lines.append(f"- {item}")
    lines.append("")
lines.append("## Tabel di balik grafik")
for name, table in saved:
    lines.append(f"### {name}.png")
    lines.append("```")
    lines.append(table.head(40).to_string(index=False))
    lines.append("```")
with open(f"{OUTPUT_FOLDER}/fakta_topik.md", "w", encoding="utf-8") as report:
    report.write("\n".join(lines))
print("\n".join(lines))